# 5. (2 points)

A Cape Code loss reserving calculation has the following inputs and estimates:

- Total premium is $10,000,000
- Estimated ELR is 65%
- Process variance/mean ratio is 50,000
- The parameter covariance matrix is:

In [1]:
import pandas as pd

covariance_matrix = pd.DataFrame(
    data={
    'ELR': [0.0029, -0.0042, 0.19],
    'ω': [-0.0042, 0.0055, -0.41],
    'θ': [0.19, -0.41, 25.52]
    },
    index= ['ELR', 'ω', 'θ']
)

covariance_matrix

,ELR,ω,θ
ELR,0.0029,-0.0042,0.19
ω,-0.0042,0.0055,-0.41
θ,0.1900,-0.4100,25.52



* a. (1 point) Calculate the coefficient of variation of prospective losses.
* b. (0.5 point) Briefly describe what process variance and parameter variance of the prospective losses measure.
* c. (0.5 point) Briefly describe whether the Cape Cod method typically has a higher or lower parameter variance than the chain ladder method.

## Solution

### a.

Source: Clark, section 4.3.1 (Variance of the Prospective Losses). `chainladder`'s
`ClarkLDF` has no process/parameter variance or covariance matrix output, so this is done by hand.

For the prospective period, expected losses are $\mu = \text{Premium} \times \text{ELR}$, so the only
parameter that matters is the ELR. $\omega$ and $\theta$ drive the reserve on past accident years, not
the prospective losses, so the rest of the covariance matrix is not used.

- Process variance = (variance/mean ratio) $\times \mu$
- Parameter variance = $\text{Premium}^2 \times \text{Var}(\text{ELR})$
- Total CV = $\sqrt{\text{process var} + \text{parameter var}} \,/\, \mu$

In [2]:
premium = 10_000_000
elr = 0.65
variance_to_mean = 50_000

expected_losses = premium * elr
process_variance = variance_to_mean * expected_losses
parameter_variance = premium**2 * covariance_matrix.loc['ELR', 'ELR']
total_variance = process_variance + parameter_variance

pd.Series({
    'Expected prospective losses': expected_losses,
    'Process variance': process_variance,
    'Parameter variance': parameter_variance,
    'Total variance': total_variance,
    'Process CV': process_variance**0.5 / expected_losses,
    'Parameter CV': parameter_variance**0.5 / expected_losses,
    'Total CV': total_variance**0.5 / expected_losses,
})

Expected prospective losses    6.500000e+06
Process variance               3.250000e+11
Parameter variance             2.900000e+11
Total variance                 6.150000e+11
Process CV                     8.770580e-02
Parameter CV                   8.284869e-02
Total CV                       1.206491e-01
dtype: float64

**a.** Total CV $\approx$ **12.1%** (process CV 8.8%, parameter CV 8.3%).

### b.

- **Process variance** measures the random variation of actual losses around their expected value, even if
  the true parameters were known. It is modeled as a constant variance/mean ratio times the expected losses.
- **Parameter variance** measures the uncertainty in the estimated expected losses because the model
  parameters (ELR, $\omega$, $\theta$) are estimated from limited data. It comes from the covariance matrix of the MLE
  estimates (the inverse of the information matrix).

### c.

The Cape Cod method typically has a **lower** parameter variance than the chain ladder (LDF) method. Cape Cod
estimates only 3 parameters (ELR, $\omega$, $\theta$), while the LDF method estimates a separate ultimate for each
accident year plus $\omega$ and $\theta$ (n + 2 parameters), so it is overparameterized. Cape Cod also uses the exposure (premium)
information, and the extra information reduces estimation error. The Cape Cod method may have somewhat
higher process variance.

## chainladder-python gaps

**What `chainladder` (0.10.1) covers.** `cl.ClarkLDF` fits Clark's LDF and Cape Cod methods. With a
`sample_weight` (premium) it exposes `elr_`, `omega_`, `theta_`, and `scale_`, which is meant to be the process
variance/mean ratio used in part a. See the bug described below: for Cape Cod, `scale_` is currently wrong.

**What is missing for this problem.**

- No parameter covariance matrix. The matrix given in the problem (ELR, $\omega$, $\theta$) is the inverse of the
  information matrix at the MLE, and `ClarkLDF` neither stores nor exposes the information matrix or its inverse.
- No process variance, parameter variance, or standard error/CV of Clark's reserve estimate, by accident year or in total.
- No variance for prospective losses (premium $\times$ ELR) or for discounted reserves (Clark sections 4.3.1 and 4.3.3).
  Variance output exists only for Mack (`MackChainladder`) and the bootstrap, not for Clark.

**Suggested feature.** Add variance outputs to `ClarkLDF`:

- `cov_`: parameter covariance matrix (from the inverse information matrix, scaled by `scale_`), indexed by parameter
  name (`ELR`, `omega`, `theta`, or per-origin ultimates for the LDF method).
- `process_variance_`, `parameter_variance_`, `total_variance_`, `std_err_` (and a CV), by origin and in total,
  mirroring the attribute names on `MackChainladder`.
- A method such as `prospective_variance(premium)` returning expected losses, process variance, parameter variance, and
  CV for a future period's premium, which would turn the calculation in part a into one line.

### Bug found: `ClarkLDF.scale_` is wrong for the Cape Cod method

While checking `ClarkLDF` against the worked examples in Clark section 4, the LDF method and the Cape Cod fit reproduced the
paper's $\omega$, $\theta$, ELR and, for the LDF method, the variance/mean ratio $\sigma^2$ (65,029). But the Cape Cod variance/mean
ratio does not match. The paper's triangle is the `genins` sample except for one cell (2001 origin, 48 months: 2,218,270 in
`genins`, 2,182,708 in the paper, apparently a transposition), which the reproduction below changes to match the paper:

| Cape Cod $\sigma^2$ | Value |
|---|---|
| `chainladder` 0.10.1 `scale_` | 53,822 |
| Recomputed from ELR $\times$ premium fits | **61,577** (matches Clark section 4.2) |

**Cause.** In `ClarkLDF.fit`, `incremental_fits_` is built from ultimates equal to the latest diagonal divided by the growth
function ($\text{ult}_i = \text{latest}_i / G(x_i)$), whatever the method. That is correct for the LDF method. For Cape Cod the fitted
incremental in each cell should be $\text{ELR} \times \text{premium}_i \times [G(\text{end}) - G(\text{start})]$, using the model's own
expected ultimate. The incremental totals are nearly the same either way (both fits sum to about 34.36M, as the ELR's maximum-likelihood condition forces), so the error is in the cell-by-cell pattern, not the total. `scale_` is a chi-square sum over `incremental_fits_`, so it is understated (by about 13% in this example),
and `norm_resid_` is wrong for Cape Cod too. The fitted parameters themselves are fine, because the optimizer uses the correct
ELR $\times$ premium model.

**Impact.** Any process variance computed from `scale_` on a Cape Cod fit is too small. This matters for part a, where the
variance/mean ratio is the input.

**Suggested fix.** When `sample_weight` is given, compute `incremental_fits_` from `elr_ * sample_weight` instead of from the
latest diagonal, and add a regression test against the Clark section 4.2 figures (the reproduction below).

In [3]:
# Reproduction: Clark section 4 example. The triangle is the `genins` sample (Mack 1993) with one cell
# changed to match the paper (2001 origin, 48 months: 2,218,270 in genins vs 2,182,708 in the paper).
# Premium is 10M in 2001 plus 400K per year.
import numpy as np
import chainladder as cl

tri = cl.load_sample("genins").copy()
tri.values[0, 0, 0, 3] = 2_182_708

tri["premium"] = tri["values"] * 0 + np.array([10_000_000 + 400_000 * i for i in range(10)])[:, None]
premium = tri["premium"].latest_diagonal

cape_cod = cl.ClarkLDF().fit(tri["values"], sample_weight=premium)
fits = cape_cod.incremental_fits_.values[0, 0]

# What the fitted incrementals should be for Cape Cod: ELR x premium x [G(end) - G(start)]
th, om, elr = (float(x.values.ravel()[0]) for x in (cape_cod.theta_, cape_cod.omega_, cape_cod.elr_))
G = lambda x: 1 / (1 + (th / x) ** om)
expected = np.full((10, 10), np.nan)
for i in range(10):
    for j in range(10 - i):
        end = (j + 1) * 12 - 6
        start = max(end - 12, 0)
        expected[i, j] = premium.values[0, 0, i, 0] * elr * (G(end) - (G(start) if start else 0))

actual = tri["values"].cum_to_incr().values[0, 0]
chi_sq = np.nansum((actual - expected) ** 2 / expected)

pd.Series({
    'ELR (paper: 59.78%)': elr,
    'scale_ from chainladder': float(cape_cod.scale_.values.ravel()[0]),
    'scale_ using ELR x premium fits': chi_sq / (55 - 3),
    'sum of chainladder fitted incrementals': np.nansum(fits),
    'sum of actual incrementals': np.nansum(actual),
    'sum of ELR x premium fitted incrementals': np.nansum(expected),
})

ELR (paper: 59.78%)                         5.977879e-01
scale_ from chainladder                     5.382178e+04
scale_ using ELR x premium fits             6.157707e+04
sum of chainladder fitted incrementals      3.435809e+07
sum of actual incrementals                  3.435809e+07
sum of ELR x premium fitted incrementals    3.435773e+07
dtype: float64